In [1]:
%cd drive/MyDrive/Thesis/Models/PLM_embedding
# %cd /content/drive/MyDrive/Thesis/Data/training_embeddings

/content/drive/MyDrive/Thesis/Models/PLM_embedding


In [ ]:
!python --version

Python 3.8.16


In [ ]:
!nvidia-smi

Sat Jan  7 10:52:50 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 460.32.03    Driver Version: 460.32.03    CUDA Version: 11.2     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla T4            Off  | 00000000:00:04.0 Off |                    0 |
| N/A   39C    P0    26W /  70W |      0MiB / 15109MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [2]:
import numpy as np
import torch

In [ ]:
# a = np.load("../../Data/all_test_chains.npz")
# a = np.load("../../Data/n62.npz")

In [5]:
seq1 = [12,  0, 10,  3, 19, 11, 11, 18, 14, 13,  9, 11, 15, 10,  3,  3,
         6, 10, 11, 11, 18,  7,  7,  7,  9, 16, 10, 16,  3, 10, 10, 10,
         2, 15, 16, 16, 10,  9,  3,  5,  0, 11,  3,  1,  9,  5, 11, 16,
        11, 15,  7,  3,  6, 10, 14,  8, 18, 19, 15, 18,  2,  6, 14, 19,
        10, 19, 13, 18, 16, 16, 10, 10, 15, 10,  0,  9, 10,  2,  3, 19,
        11, 10,  9,  1,  1, 18,  0, 18,  0,  6,  0, 11,  5, 13,  1, 15,
        10, 10,  8, 16,  6,  2,  6,  6,  2, 10, 10,  6,  9, 15, 11, 10,
        10,  3, 10, 11,  9,  2,  1,  4,  3, 14,  9, 11, 13, 18, 10,  6,
        11, 11,  1,  1,  9,  9,  5, 11,  6, 13,  4, 19,  8, 13,  9,  3,
        18, 10, 11, 18, 16, 11,  3, 10, 11,  6,  3, 17, 11, 10, 15,  7,
         5,  9, 10,  8, 11,  7, 18, 19, 18, 10,  3, 11,  2,  5, 10,  9,
         7, 10,  9,  0,  6, 15,  9, 11, 15, 11,  9, 19,  6, 12,  9,  1,
        14, 10,  2, 10, 11,  6,  9, 14,  6, 11, 10, 11, 15, 10,  9,  6,
         1,  1,  7,  9]

seq2 = [ 7, 15, 15,  7, 15, 15,  7,  7, 16, 16,  4, 14, 14, 14, 19, 15,
         9,  6,  8,  0,  3,  9,  1, 19, 11,  2, 18, 15, 19,  2, 15,  1,
         6,  1, 18, 19,  4,  2, 15,  7, 13, 11,  1, 11,  0,  7, 16, 15,
        16, 10,  9,  6,  4, 19,  9,  2, 11,  2, 16,  2, 19,  0,  8, 17,
        16, 16, 14, 15, 10, 11,  4,  9,  1,  3, 14, 15, 10,  0]

In [7]:
seq1 = [reverse_dict[x] for x in seq1]
seq2 = [reverse_dict[x] for x in seq2]

In [12]:
seq1 = "".join(seq1)
seq2 = "".join(seq2)

In [6]:
AA_dict = {i:j for i,j in zip(['A','R','N','D','C','Q','E','G','H','I','L','K','M','F','P','S','T','W','Y','V','X'],range(21))}
reverse_dict = {j:i for i,j in zip(['A','R','N','D','C','Q','E','G','H','I','L','K','M','F','P','S','T','W','Y','V','X'],range(21))}
# esm_dict = {i:j for i,j in zip(list("LAGVSERTIDPKQNFYMHWC"),range(4,24))}
# convert = {AA_dict[x]:esm_dict[x] for x in list("LAGVSERTIDPKQNFYMHWC")}
# replace = np.arange(0,21)
# replace[list(convert.keys())] = list(convert.values())


In [8]:
model, alphabet = torch.hub.load("facebookresearch/esm:main", "esm2_t33_650M_UR50D")
batch_converter = alphabet.get_batch_converter()
model = model.to(device="cuda")
model.eval() 

Downloading: "https://github.com/facebookresearch/esm/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt


ESM2(
  (embed_tokens): Embedding(33, 1280, padding_idx=1)
  (layers): ModuleList(
    (0): TransformerLayer(
      (self_attn): MultiheadAttention(
        (k_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (v_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (q_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (out_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (rot_emb): RotaryEmbedding()
      )
      (self_attn_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
      (fc1): Linear(in_features=1280, out_features=5120, bias=True)
      (fc2): Linear(in_features=5120, out_features=1280, bias=True)
      (final_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    )
    (1): TransformerLayer(
      (self_attn): MultiheadAttention(
        (k_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (v_proj): Linear(in_features=1280, out_features=1280, bia

In [9]:
def get_embeddings(folder, data):
  batch_labels, batch_strs, batch_tokens = batch_converter(data)
  batch_tokens = batch_tokens.to(device="cuda")
  results = model(batch_tokens, repr_layers=[33], return_contacts=False)
  results = results["representations"][33]

  for j in range(len(data)):
    chain = data[j][0]
    seq   = data[j][1]
    seqlen = len(seq)
    torch.save(results[j][:seqlen+2].detach().cpu(), folder+chain+'.pt')
  torch.cuda.empty_cache()

In [13]:
with torch.no_grad():
  # for i in range(len(a.files)//5):
    # chains = a.files[5*i : 5*(i+1)]
    # seqs   = []
    # for chain in chains:
      # seqs.append("".join([reverse_dict[x] for x in a[chain][0]]))

    data = [("1ZT2_B",seq1), ("2PSM_C",seq2)]
    # data = [*zip(chains,seqs)]

    get_embeddings("../../Data/n2/", data)
    # get_embeddings("../../Data/test_embeddings")

In [ ]:
!ls ../../Data/test_embeddings | wc -l

18280


In [ ]:
# with torch.no_grad():
  # for i in range(560, 21008//10): #get embeddings from training data in batches. 21008 chains in train set
  # ESM uses 0 for BOS, 2 for EOS, 1 for end padding. Add extra position for BOS, EOS
  # for i in range(21008//5):
    # embedding_dict = {}
    # chains = a.files[10 * i : 10 * (i+1)]
    # if i == (21008//10) - 1:
    #   chains = a.files[10 * i:]

    # chains = a.files[5 * i : 5 * (i+1)]
    # seqs   = []
    # for chain in chains:
    #   seqs.append("".join([reverse_dict[x] for x in a[chain][0]]))
    # data = [*zip(chains,seqs)]

    # batch_labels, batch_strs, batch_tokens = batch_converter(data)
    # batch_tokens = batch_tokens.to(device="cuda")

    # batch_lens = (batch_tokens != alphabet.padding_idx).sum(1)
    # batch_tokens = np.full((5,700),1)
    # for idx, chain in enumerate(chains):
    #   seq = a[chain][0]
    #   if len(seq) < 699:
    #     x[idx][0] = 0 
    #     x[idx][1:len(seq) + 1] = replace[seq]
    #     x[idx][len(seq) + 1]   = 2
    #   else:
    #     x[idx][0] = 0
    #     x[idx][1:] = seq[0:699]

    # # batch_tokens = torch.from_numpy(x).to(device="cuda")
    # # batch_tokens = batch_tokens.view(5,700)
    # results = model(batch_tokens, repr_layers=[33], return_contacts=False)
    # results = results["representations"][33]

    # for j in range(5):
    #   seqlen = len(seqs[j])
    #   torch.save(results[j][:seqlen+2].detach().cpu(), "../../Data/training_embeddings/"+chains[j]+'.pt')
        # torch.save( results, "../../Data/training_embeddings/test1.pt") #save each batch independently, otherwise willrun out of RAM

    # np.save("../../Data/training_embeddings/test1.np", results.cpu.detach().numpy())
    # batch_tokens = np.full((5,700), 1)
    # for j in range(5):
    #   seq = a[chains[j]][0]
    #   batch_tokens[j][0] = 0
    #   batch_tokens[j][1:len(seq)]
    # x = np.full(700,1)
    # x[0] = 0
    # x[1:len(np_seq)+1] = replace[np_seq]
    # x[len(np_seq)+1] = 2
    # results = model(batch_tokens, repr_layers=[33], return_contacts=False)
    # results = results["representations"]
    # # embedding_dict = {i:j for i,j in zip(chains, results)}
    # torch.save( results, "../../Data/training_embeddings/training_embeddings"+str(i)+".pt") #save each batch independently, otherwise willrun out of RAM
    # torch.cuda.empty_cache()

In [ ]:
# torch.cuda.empty_cache()
# torch.no_grad()
# model = model.to(device="cuda")
# chains = a.files[0:10]
# np_seq = [a[x][0] for x in chains]
# seqs = ["".join([AA_dict[x] for x in y]) for y in np_seq]
# # data = [*zip(chains,seqs)]
# batch_converter = alphabet.get_batch_converter()
# batch_labels, batch_strs, batch_tokens = batch_converter(data)
# batch_tokens = batch_tokens.to(device="cuda")
# results = model(batch_tokens, repr_layers=[33], return_contacts=False)

222
